# 🏀 Proyecto Machine Learning - Draft NBA 2026 🏀

## ¿Qué es el Draft de la NBA?

El **Draft de la NBA** es el proceso anual mediante el cual los 30 equipos de la NBA seleccionan jugadores de universidad, ligas europeas y otras competiciones. Consta de 2 rondas y 60 picks.

La posición en la que eres drafteado determina tu contrato, tu equipo y en muchos casos tu carrera entera.

---
## Punto de partida

Cada año, los equipos NBA seleccionan jugadores en el Draft. Las predicciones tradicionales se basan en opiniones de analistas o de profesionales dedicados a seguir la trayectoria de jóvenes talentos. Sin embargo, me he querido preguntar:

¿Podemos **predecir con Machine Learning** si un jugador será drafteado, en qué **ronda** y en qué **rango de pick**? En concreto, ¿qué resultados obtendrían los españoles Aday Mara, Baba Miller y Sergio De Larrea?

---
## Fundamentos del proyecto

### 1. Definir las predicciones

- **Clasificación de ronda**: ¿R1 / R2 / No drafteado?
- **Clasificación de rango de pick**: ¿1-10 / 11-20 / 21-30 / 31-40 / 41-50 / 51-60?
- **Clustering**: ¿A qué arquetipo de jugador pertenece? (base explosivo, alero 3&D, pívot clásico...)

### 2. ¿Por qué es interesante este proyecto?

Las predicciones actuales se basan en **opiniones subjetivas** de scouts y analistas. Este proyecto propone un enfoque basado en **datos objetivos**: estadísticas reales de rendimiento en cancha.

### 3. ¿A quién le interesa?

Además de al propio aficionado al baloncesto, también puede llamar la atención de un **medio de comunicación deportivo** que quiere publicar un artículo sobre las posibilidades de los jugadores españoles en el Draft 2026 respaldado no por opiniones, sino por un modelo de Machine Learning.

---

---
## Fuente de datos

### Dataset — Estadísticas universitarias NCAA (2009–2021)
- Más de **61.000 registros** de jugadores de la NCAA entre 2009 y 2021.
- Una fila por jugador por temporada.
- Variables: puntos, rebotes, asistencias, tapones, robos, eficiencia de tiro (eFG%, TS%), BPM, usage rate y muchas más.
- **Incluye directamente el número de pick** del Draft para los jugadores seleccionados — es nuestra variable objetivo.
- Fuente: Kaggle (`College_BasketballPlayers2009-2021`).

### ¿Por qué este dataset?

Nos da las **variables de rendimiento** (lo que hace el jugador en cancha) y la **variable objetivo** (qué pick tuvo en el Draft). Es todo lo que necesitamos para entrenar el modelo.

Los tres españoles no tienen stats NCAA — son europeos. Para predecirlos construiremos manualmente un DataFrame con sus estadísticas reales de ACB y EuroLeague.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ncaa = pd.read_csv('../datos/College_BasketballPlayers2009-2021.csv', low_memory=False)

print(f"NCAA histórico: {ncaa.shape[0]:,} registros | {ncaa.shape[1]} variables")
print(f"Jugadores únicos: {ncaa['player_name'].nunique():,}")
print(f"Media de temporadas por jugador: {len(ncaa) / ncaa['player_name'].nunique():.1f}")
print(f"Años disponibles: {sorted(ncaa['year'].unique())}")
print()
print(f"Jugadores drafteados con pick conocido: {ncaa['pick'].notna().sum():,}")
print(f"Jugadores no drafteados: {ncaa['pick'].isna().sum():,}")

---

## Los tres españoles

Los tres candidatos que queremos predecir participaron en el NBA Draft Combine 2026. Sus estadísticas se introducirán manualmente desde sus ligas de origen (ACB / EuroLeague) para alimentar el modelo en el momento de la predicción.

| Jugador | Posición | Liga | Mock Draft |
|---------|----------|------|------------|
| Aday Mara | C | ACB | ~Top 10 |
| Baba Miller | PF/SF | NCAA | ~Pick 45 |
| Sergio De Larrea | PG | EuroLeague / Valencia Basket | ~Pick 40 |

> **Nota:** Los españoles no tienen stats NCAA. El modelo se entrena con datos históricos de la NCAA y la predicción se realiza introduciendo sus estadísticas reales de forma manual.

---

## Limpieza del dataset

Antes de construir el dataset de entrenamiento, realizamos las siguientes operaciones de limpieza:

### 1. Tratamiento de NaN
- **`pick`**: no se toca — los NaN son los jugadores no drafteados (clase ND).
- **`Rec Rank`** (69% NaN): eliminada.
- **`rimmade/(rimmade+rimmiss)` y `midmade/(midmade+midmiss)`**: imputadas con la mediana por año. El año 2009 no tenía datos — imputado con la mediana global.
- **`ast/tov`**: imputada con mediana por año.
- **`yr`, `ht`**: imputadas con la moda.
- **Stats básicas** (`pts`, `ast`, `treb`...): imputadas con mediana por año.
- **Columnas eliminadas**: `Rec Rank`, `num`, `Unnamed: 65`, `type`, `dunksmade/(dunksmade+dunksmiss)`, totales brutos de tiro.

### 2. Parseo de altura
La columna `ht` estaba en formato corrupto por Excel (ej: `"7-Jun"` en lugar de `6'7"`). Se parseó a centímetros creando la columna `altura_cm`. Los valores imposibles se imputaron con la mediana.

### 3. Outliers
- **`eFG` > 100**: 120 filas con valores imposibles eliminadas.
- **`eFG` = 0**: reemplazado con la mediana de jugadores con eFG > 0.

---

## Construcción del dataset de entrenamiento

### Decisiones clave

**1. Una fila por jugador — última temporada**  
El dataset original tiene una fila por jugador por temporada (media: 2.4 temporadas). Para evitar que el modelo vea al mismo jugador varias veces, nos quedamos solo con la última temporada antes del draft.

**2. Balanceo de clases**  
El 97.6% de los registros son jugadores no drafteados (ND). Un modelo entrenado con ese desbalance aprendería a decir siempre ND sin aprender nada útil. Estrategia:
- R1 y R2: todos los disponibles.
- ND: top 1.500 jugadores por BPM con mínimo de 10 minutos por partido — los mejores que no llegaron al draft.

**3. Dos targets**  
- `ronda`: R1 / R2 / ND
- `rango_pick`: 1-10 / 11-20 / 21-30 / 31-40 / 41-50 / 51-60 / ND

**4. Reducción de features**  
De 66 columnas originales a 37 features — eliminando identificadores, métricas de equipo, columnas redundantes y valores absolutos cuando ya existe el porcentaje equivalente.

In [ ]:
datos_draft = pd.read_csv('../datos/ncaa_modelo.csv')

print(f"Dataset final: {datos_draft.shape[0]} jugadores | {datos_draft.shape[1]} features")
print()
print("Distribución target ronda:")
print(datos_draft['ronda'].value_counts())
print()
print("Distribución target rango_pick:")
print(datos_draft['rango_pick'].value_counts().sort_index())

---

## Modelos planificados

### 1. Clasificación — ¿En qué ronda será drafteado?
**Target:** R1 / R2 / ND  
**Algoritmo:** por determinar tras el EDA (XGBoost, SVM con kernel RBF, LightGBM)  
**Features:** 37 estadísticas NCAA

### 2. Clasificación — ¿En qué rango de pick?
**Target:** 1-10 / 11-20 / 21-30 / 31-40 / 41-50 / 51-60 / ND  
**Algoritmo:** por determinar  
**Features:** 37 estadísticas NCAA

### 3. Clustering — ¿A qué arquetipo pertenece?
**Objetivo:** agrupar jugadores en perfiles tipo (base explosivo, alero 3&D, pívot clásico...)  
**Algoritmo:** K-Means  
**Features:** estadísticas de cancha

---

## Objetivo final

- Un **análisis completo** sobre las posibilidades reales de Aday Mara, Baba Miller y Sergio De Larrea en el Draft 2026, respaldado por Machine Learning.
- Una **app interactiva** donde introducir las estadísticas de cualquier jugador y obtener: ¿será drafteado?, ¿en qué rango?, ¿a qué tipo de jugador se parece?
- Acercar el periodismo deportivo a herramientas tecnológicas poco utilizadas en el sector.